# 02. Vector Validation — 수정판

**실행 순서**
1. `AXIS="economic"` → Cell 1~9
2. `AXIS="social"` → Cell 1~9 (커널 재시작)
3. Cell 10~11 (common layer + cosine) — 두 축 완료 후 한 번

***주의: economic / social 각각 별도의 system prompts.json로드되니 폴더에 저장되어 있어야 함
output/qwen2.5-7b-instruct/economic/system_prompts.json
output/qwen2.5-7b-instruct/social/system_prompts.json

In [ ]:
# ── Cell 1: 설정 ────────────────────────────────────────────────
import os
from pathlib import Path
# --- repo-relative project root: works on any machine ---
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

MODEL_KEY = "qwen"     # "llama" / "qwen"
AXIS      = "social"  # "economic" / "social"

VECTOR_TYPE           = "response_avg"
JUDGE_MODEL           = "gpt-4.1-mini-2025-04-14"
MAX_CONCURRENT_JUDGES = 50
N_ROLLOUTS            = 10
MAX_NEW_TOKENS        = 512
# 음수 포함 대칭 sweep (양방향 제어 검증)
ALPHAS_STEER = [-0.40, -0.30, -0.20, -0.15, -0.10, -0.05,
                  0.0,
                  0.05,  0.10,  0.15,  0.20,  0.30,  0.40]

MODEL_CONFIGS = {
    "llama": {"model_id": "meta-llama/Llama-3.1-8B-Instruct",
              "short_name": "llama-3.1-8b-instruct", "trust_remote_code": False},
    "qwen":  {"model_id": "Qwen/Qwen2.5-7B-Instruct",
              "short_name": "qwen2.5-7b-instruct",  "trust_remote_code": True},
}
AXIS_CONFIGS = {
    "economic": {"trait_name": "economic_left",        "trait_label": "Economic Left"},
    "social": {"trait_name": "social_progressive", "trait_label": "Social Progressive"},
}
cfg = MODEL_CONFIGS[MODEL_KEY]
axis_cfg = AXIS_CONFIGS[AXIS]
TRAIT_NAME, TRAIT_LABEL = axis_cfg["trait_name"], axis_cfg["trait_label"]
print(f"{cfg['model_id']} / {AXIS} / {VECTOR_TYPE}")

In [ ]:
# ── Cell 2: 환경 ────────────────────────────────────────────────
import json, math, asyncio
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from scipy.stats import pearsonr
import nest_asyncio
nest_asyncio.apply()

try:
    from dotenv import load_dotenv; load_dotenv()
except ImportError: pass

assert os.environ.get("OPENAI_API_KEY")
hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    from huggingface_hub import login; login(hf_token)

assert torch.cuda.is_available()
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Cell 3: 벡터 & 데이터 로드 (metadata 검증) ──────────────────
vec_dir = Path("persona_vectors") / cfg["short_name"]

vec_all = torch.load(vec_dir / f"{TRAIT_NAME}_{VECTOR_TYPE}_diff.pt", weights_only=False)
info = torch.load(vec_dir / f"{TRAIT_NAME}_{VECTOR_TYPE}_best_layer_info.pt",
                  weights_only=False)
assert info["model"] == cfg["short_name"], f"모델 불일치: {info['model']}"
assert info["vector_type"] == VECTOR_TYPE,  f"벡터타입 불일치: {info['vector_type']}"
assert info["trait_name"] == TRAIT_NAME

BEST_LAYER = info["best_layer"]
print(f"best_layer({AXIS}) = {BEST_LAYER}")

with open(f"data_generation/trait_data_eval/{TRAIT_NAME}.json") as f:
    td = json.load(f)
EVAL_PROMPT = td["eval_prompt"].replace("{{question}}", "{question}").replace("{{answer}}", "{answer}")
EVAL_QUESTIONS = td["questions"]
assert "{question}" in EVAL_PROMPT
print(f"eval 문항: {len(EVAL_QUESTIONS)}")

In [ ]:
# ── Cell 4: 모델 로드 ───────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

tokenizer = AutoTokenizer.from_pretrained(
    cfg["model_id"], trust_remote_code=cfg["trust_remote_code"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    cfg["model_id"], torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"}, trust_remote_code=cfg["trust_remote_code"])
model.eval()
print("로드 완료")

In [ ]:
# ── 공통 헬퍼 (모든 노트북 동일) ─────────────────────────────────
# 수정사항 반영:
#   (1) add_special_tokens=False 통일 (Llama BOS 중복 방지)
#   (2) generation slicing: input_ids.shape[1] 기준 (left padding 대응)
#   (3) unit_vector: 벡터 norm 통제
#   (4) dual-basis projection: 비직교 2축 좌표 정확 계산

def tokenize_chat_batch(messages_batch):
    """chat 배치 -> (prompt_texts, inputs). add_special_tokens=False 통일."""
    prompt_texts = [
        tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        for messages in messages_batch
    ]
    inputs = tokenizer(
        prompt_texts, return_tensors="pt",
        padding=True, truncation=True,
        add_special_tokens=False,
    ).to(device)
    return prompt_texts, inputs


def tokenize_chat_single(messages):
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(
        prompt_text, return_tensors="pt",
        add_special_tokens=False,
    ).to(device)
    return prompt_text, inputs


def decode_generated_batch(output_ids, inputs):
    """left padding과 무관하게 새로 생성된 토큰만 디코딩."""
    input_width = inputs["input_ids"].shape[1]
    return tokenizer.batch_decode(
        output_ids[:, input_width:], skip_special_tokens=True)


def unit_vector(v, eps=1e-12):
    v = v.float()
    return v / v.norm().clamp_min(eps)


def make_dual_projector(v_econ, v_cult, cond_limit=100.0):
    """비직교 2축의 dual-basis projector. coords = P @ h"""
    u_e = unit_vector(v_econ)
    u_c = unit_vector(v_cult)
    basis = torch.stack([u_e, u_c], dim=1)          # (hidden, 2)
    gram = basis.T @ basis                            # (2, 2)
    cond = float(torch.linalg.cond(gram))
    assert cond < cond_limit, f"ill-conditioned basis: cond={cond:.1f}"
    return torch.linalg.solve(gram, basis.T)          # (2, hidden)


def project_2d(hidden_state, dual_projector):
    coords = dual_projector @ hidden_state.float()
    return float(coords[0]), float(coords[1])


class LastTokenSteerer:
    """마지막 토큰 위치에 alpha * original_hidden_norm * unit_vector 주입."""
    def __init__(self, model, vector, alpha, layer_1based):
        self.model = model
        self.alpha = float(alpha)
        self.layer_idx = layer_1based - 1
        self.vec = unit_vector(vector)
        self._handle = None

    def _hook(self, module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        h = h.clone()

        v = self.vec.to(device=h.device, dtype=h.dtype)

        original_norm = (
            h[:, -1, :]
            .norm(dim=-1, keepdim=True)
            .clamp_min(1e-8)
        )

        h[:, -1, :] += self.alpha * original_norm * v

        return (h,) + out[1:] if isinstance(out, tuple) else h

    def __enter__(self):
        self._handle = self.model.model.layers[
            self.layer_idx
        ].register_forward_hook(self._hook)
        return self

    def __exit__(self, *_):
        if self._handle is not None:
            self._handle.remove()
            self._handle = None


print("공통 헬퍼 정의 완료 (add_special_tokens=False / input_width slicing / "
      "unit vector / dual-basis / last-token steering)")

In [ ]:
import asyncio
import math
from openai import InternalServerError, APITimeoutError, RateLimitError

class OpenAiJudge:
    """0-100 judge, logprob-weighted (Betley et al. 2025)."""
    def __init__(self, model_name, prompt_template):
        from openai import AsyncOpenAI
        self.client = AsyncOpenAI()
        self.model = model_name
        self.prompt_template = prompt_template

    async def __call__(self, **kwargs):
        messages = [{"role": "user",
                     "content": self.prompt_template.format(**kwargs)}]
        for attempt in range(6):
            try:
                c = await self.client.chat.completions.create(
                    model=self.model, messages=messages,
                    max_tokens=1, temperature=0,
                    logprobs=True, top_logprobs=20, seed=0)
                break
            except (InternalServerError, APITimeoutError, RateLimitError):
                if attempt == 5:
                    raise
                await asyncio.sleep(2 ** attempt)

        # 이후 파싱 로직
        try:
            lp = c.choices[0].logprobs.content[0].top_logprobs
        except (IndexError, AttributeError):
            return None
        probs = {el.token: float(math.exp(el.logprob)) for el in lp}
        total = s = 0.0
        for k, v in probs.items():
            try:
                ik = int(k)
            except ValueError:
                continue
            if 0 <= ik <= 100:
                s += ik * v; total += v
        return None if total < 0.25 else s / total


async def score_batch(judge, questions, answers, sem):
    async def _one(q, a):
        async with sem:
            return await judge(question=q, answer=a)
    return await asyncio.gather(*[_one(q, a) for q, a in zip(questions, answers)])


@torch.inference_mode()
def generate_steered(questions, layer_1based, vector, alpha,
                     max_new_tokens=512, do_sample=False, seed=None):
    msgs = [[{"role": "user", "content": q}] for q in questions]
    _, inputs = tokenize_chat_batch(msgs)
    if seed is not None:
        torch.manual_seed(seed)
    gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample,
                      pad_token_id=tokenizer.pad_token_id,
                      eos_token_id=tokenizer.eos_token_id)
    if do_sample:
        gen_kwargs["temperature"] = 1.0
    gen_cfg = GenerationConfig(**gen_kwargs)
    if abs(alpha) < 1e-9:
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            out = model.generate(**inputs, generation_config=gen_cfg)
    else:
        with LastTokenSteerer(model, vector, alpha, layer_1based):
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                out = model.generate(**inputs, generation_config=gen_cfg)
    return decode_generated_batch(out, inputs)


@torch.inference_mode()
def generate_with_system_batch(system_prompt, questions,
                               max_new_tokens=512, do_sample=False, seed=None):
    msgs = []
    for q in questions:
        m = []
        if system_prompt:
            m.append({"role": "system", "content": system_prompt})
        m.append({"role": "user", "content": q})
        msgs.append(m)
    _, inputs = tokenize_chat_batch(msgs)
    if seed is not None:
        torch.manual_seed(seed)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample,
                          pad_token_id=tokenizer.pad_token_id,
                          eos_token_id=tokenizer.eos_token_id)
        if do_sample:
            gen_kwargs["temperature"] = 1.0
        out = model.generate(**inputs,
                             generation_config=GenerationConfig(**gen_kwargs))
    return decode_generated_batch(out, inputs)


@torch.inference_mode()
def get_prompt_last_projection_batch(system_prompt, questions, layer, vector_unit):
    """left padding에서 마지막 실제 토큰은 항상 index -1."""
    msgs = []
    for q in questions:
        m = []
        if system_prompt:
            m.append({"role": "system", "content": system_prompt})
        m.append({"role": "user", "content": q})
        msgs.append(m)
    _, inputs = tokenize_chat_batch(msgs)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        out = model(**inputs, output_hidden_states=True)
    hs = out.hidden_states[layer][:, -1, :].float().cpu()   # (B, hidden)
    del out
    return (hs @ vector_unit.float()).tolist()


judge = OpenAiJudge(JUDGE_MODEL, EVAL_PROMPT)
sem = asyncio.Semaphore(MAX_CONCURRENT_JUDGES)
out_dir = Path("output") / cfg["short_name"] / AXIS
out_dir.mkdir(parents=True, exist_ok=True)
print("유틸 정의 완료")

In [ ]:
# ── Cell 7: A. Token Position 비교 (부록용 보조 분석) ───────────
# VECTOR_TYPE은 prompt_avg로 사전 고정. 이 셀은 세 벡터의 unit-vector
# steering 효과를 나란히 보고하는 부록용 비교이며 선택에 쓰이지 않음.
vec_ra_L = torch.load(vec_dir / f"{TRAIT_NAME}_response_avg_diff.pt", weights_only=False)[BEST_LAYER]
vec_pa_L = torch.load(vec_dir / f"{TRAIT_NAME}_prompt_avg_diff.pt",   weights_only=False)[BEST_LAYER]
vec_pl_L = torch.load(vec_dir / f"{TRAIT_NAME}_prompt_last_diff.pt",  weights_only=False)[BEST_LAYER]

TEST_QUESTIONS = EVAL_QUESTIONS[:5]
TEST_ALPHAS = [-0.20, 0.20]

rows_vt = []
for vt_name, vec in [("response_avg", vec_ra_L),
                     ("prompt_avg", vec_pa_L),
                     ("prompt_last", vec_pl_L)]:
    for alpha in TEST_ALPHAS:
        answers = generate_steered(TEST_QUESTIONS, BEST_LAYER, vec, alpha,
                                   max_new_tokens=256)
        scores = asyncio.get_event_loop().run_until_complete(
            score_batch(judge, TEST_QUESTIONS, answers, sem))
        mean_s = float(pd.Series(scores).dropna().mean())
        rows_vt.append({"vector_type": vt_name, "alpha": alpha, "mean_score": mean_s})
        print(f"  {vt_name:14s} α={alpha:+.1f}: {mean_s:.1f}")

df_vt = pd.DataFrame(rows_vt)
df_vt.to_csv(out_dir / "token_position_comparison.csv", index=False)
print(f"저장: {out_dir}/token_position_comparison.csv (부록용)")

In [ ]:
# ── Cell 8: B. Steering Validation (음수 포함 α sweep) ──────────
STEER_VEC = torch.load(vec_dir / f"{TRAIT_NAME}_{VECTOR_TYPE}_diff.pt",
                       weights_only=False)[BEST_LAYER]

rows_steer = []
for alpha in tqdm(ALPHAS_STEER, desc="α sweep"):
    answers = generate_steered(EVAL_QUESTIONS, BEST_LAYER, STEER_VEC, alpha)
    scores = asyncio.get_event_loop().run_until_complete(
        score_batch(judge, EVAL_QUESTIONS, answers, sem))
    for q, a, s in zip(EVAL_QUESTIONS, answers, scores):
        rows_steer.append({"alpha": alpha, "question": q,
                           "answer": a, "score": s})

df_steer = pd.DataFrame(rows_steer)
df_steer.to_csv(out_dir / 
                "steering_validation.csv", index=False)

means = df_steer.groupby("alpha")["score"].mean()
print(means.round(1))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(means.index, means.values, "o-", color="steelblue", linewidth=2)
ax.axhline(50, color="gray", linestyle="--", alpha=0.5, label="Neutral")
ax.axvline(0, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Steering α (unit vector, last-token)")
ax.set_ylabel(f"{TRAIT_LABEL} Score")
ax.set_title(f"Steering Validation — {TRAIT_LABEL}\n"
             f"{cfg['short_name']} | layer={BEST_LAYER} | {VECTOR_TYPE}")
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig(out_dir / "steering_validation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Cell 9: C. Monitoring FAST (동일 계산 / Judge 병렬화) ─────────
# 160 unit (8 prompts × 20 questions), rollout 평균 후 분석.
# 통계: question fixed-effect OLS + question-clustered robust SE.
#
# 기존과 동일:
# - 동일 prompt
# - 동일 question
# - 동일 N_ROLLOUTS
# - 동일 generation seed
# - 동일 generation 방식
# - 동일 LLM judge
# - 동일 rollout 평균
# - 동일 통계
#
# 변경:
# - 각 rollout마다 judge를 기다리지 않음
# - 모든 응답을 먼저 생성한 뒤 전체를 한 번에 async judge
# → MAX_CONCURRENT_JUDGES를 실제로 충분히 활용

sp_path = out_dir / "system_prompts.json"
assert sp_path.exists(), f"system_prompts.json 필요: {sp_path}"

with open(sp_path) as f:
    MON_PROMPTS = json.load(f)

print(f"프롬프트 {len(MON_PROMPTS)}개 "
      f"(P0=suppress → P{len(MON_PROMPTS)-1}=promote)")

MON_VEC_UNIT = unit_vector(
    torch.load(
        vec_dir / f"{TRAIT_NAME}_{VECTOR_TYPE}_diff.pt",
        weights_only=False
    )[BEST_LAYER]
)

# ================================================================
# 1) PROJECTION
# deterministic — 기존 계산 그대로
# ================================================================

rows_proj = []

for p_idx, sys_p in enumerate(
    tqdm(MON_PROMPTS, desc="projection")
):
    projs = get_prompt_last_projection_batch(
        sys_p,
        EVAL_QUESTIONS,
        BEST_LAYER,
        MON_VEC_UNIT
    )

    for q, pr in zip(EVAL_QUESTIONS, projs):
        rows_proj.append({
            "prompt_idx": p_idx,
            "question": q,
            "projection": pr
        })


# ================================================================
# 2) GENERATION
#
# 중요:
# 기존과 정확히 같은 단위로 generate 호출.
#
# prompt × rollout마다
# EVAL_QUESTIONS 20개를 하나의 GPU batch로 생성.
#
# seed도 기존과 완전히 동일.
#
# 단, 여기서는 judge하지 않고 답변만 모음.
# ================================================================

rows_ans = []

total_generations = len(MON_PROMPTS) * N_ROLLOUTS

with tqdm(total=total_generations, desc="generation") as pbar:
    for p_idx, sys_p in enumerate(MON_PROMPTS):
        for r in range(N_ROLLOUTS):

            seed = 2000 + p_idx * 100 + r

            answers = generate_with_system_batch(
                sys_p,
                EVAL_QUESTIONS,
                max_new_tokens=256,
                do_sample=True,
                seed=seed
            )

            for q, a in zip(EVAL_QUESTIONS, answers):
                rows_ans.append({
                    "prompt_idx": p_idx,
                    "question": q,
                    "rollout": r,
                    "answer": a
                })

            pbar.update(1)


print(
    f"generation 완료: {len(rows_ans)} responses "
    f"({len(MON_PROMPTS)} prompts × "
    f"{len(EVAL_QUESTIONS)} questions × "
    f"{N_ROLLOUTS} rollouts)"
)


# ================================================================
# 3) JUDGE — 전체 응답 한 번에 병렬 scoring
#
# 기존:
# 매 rollout마다 20개 judge → 완료될 때까지 대기
#
# 변경:
# 모든 응답을 score_batch에 한 번에 전달.
#
# 개별 응답에 적용되는 judge 계산 자체는 동일.
# ================================================================

print(
    f"LLM Judge scoring {len(rows_ans)} responses "
    f"(concurrency={MAX_CONCURRENT_JUDGES})..."
)

judge_questions = [
    row["question"]
    for row in rows_ans
]

judge_answers = [
    row["answer"]
    for row in rows_ans
]

scores = asyncio.get_event_loop().run_until_complete(
    score_batch(
        judge,
        judge_questions,
        judge_answers,
        sem
    )
)

assert len(scores) == len(rows_ans)

for row, score in zip(rows_ans, scores):
    row["score"] = score


# ================================================================
# RAW SAVE
# ================================================================

df_proj = pd.DataFrame(rows_proj)
df_ans = pd.DataFrame(rows_ans)

df_ans.to_csv(
    out_dir / "monitoring_rollouts_raw.csv",
    index=False
)

print(
    f"raw rollouts 저장: "
    f"{out_dir / 'monitoring_rollouts_raw.csv'}"
)


# ================================================================
# 4) ROLLOUT 평균 → 160 units
# ================================================================

df_score_unit = (
    df_ans
    .groupby(
        ["prompt_idx", "question"]
    )["score"]
    .mean()
    .reset_index()
)

df_unit = df_proj.merge(
    df_score_unit,
    on=["prompt_idx", "question"],
    validate="one_to_one"
)

expected_n = (
    len(MON_PROMPTS)
    *
    len(EVAL_QUESTIONS)
)

assert len(df_unit) == expected_n, (
    f"Expected {expected_n} units, "
    f"got {len(df_unit)}"
)

df_unit.to_csv(
    out_dir / "monitoring_units.csv",
    index=False
)


# ================================================================
# 5) STATISTICS
# 단순 상관 + question-FE clustered OLS
# ================================================================

valid = df_unit.dropna(
    subset=["projection", "score"]
)

r_unit, p_unit = pearsonr(
    valid["projection"],
    valid["score"]
)

print(
    f"\n[{len(valid)}-unit] "
    f"r={r_unit:.3f}, "
    f"p={p_unit:.2e}"
)

import statsmodels.formula.api as smf

ols = smf.ols(
    "score ~ projection + C(question)",
    data=valid
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": valid["question"]
    }
)

beta = ols.params["projection"]
pval = ols.pvalues["projection"]

print(
    f"[question-FE, clustered SE] "
    f"beta={beta:.2f}, "
    f"p={pval:.2e}"
)


# ================================================================
# 6) VISUALIZATION
# ================================================================

mon_agg = (
    df_unit
    .groupby("prompt_idx")[
        ["projection", "score"]
    ]
    .mean()
    .reset_index()
)

n_p = len(MON_PROMPTS)

fig, ax = plt.subplots(
    figsize=(10, 7)
)

sc = ax.scatter(
    df_unit["projection"],
    df_unit["score"],
    c=df_unit["prompt_idx"],
    cmap="plasma",
    s=40,
    alpha=0.55,
    vmin=0,
    vmax=n_p - 1
)

plt.colorbar(
    sc,
    ax=ax,
    label=(
        f"Prompt index "
        f"(0=suppress → {n_p-1}=promote)"
    )
)

for _, row in mon_agg.iterrows():
    ax.scatter(
        row["projection"],
        row["score"],
        c=[row["prompt_idx"]],
        cmap="plasma",
        vmin=0,
        vmax=n_p - 1,
        s=300,
        edgecolors="black",
        linewidths=1.5,
        zorder=4
    )

from scipy.stats import linregress

sl, ic, _, _, _ = linregress(
    valid["projection"],
    valid["score"]
)

xs = np.linspace(
    valid["projection"].min(),
    valid["projection"].max(),
    100
)

ax.plot(
    xs,
    sl * xs + ic,
    color="gray",
    linestyle="--",
    alpha=0.7
)

ax.set_xlabel(
    f"{TRAIT_LABEL} Projection "
    f"(prompt-last, unit vector)"
)

ax.set_ylabel(
    f"{TRAIT_LABEL} Score "
    f"(rollout mean)"
)

ax.set_title(
    f"System Prompt Monitoring — {cfg['short_name']}\n"
    f"{TRAIT_LABEL} | {len(valid)} units | "
    f"r={r_unit:.3f} | "
    f"FE-clustered p={pval:.2e}"
)

ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    out_dir / "monitoring_projection.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 7) SAVE STATS
# ================================================================

with open(
    out_dir / "monitoring_stats.json",
    "w"
) as f:
    json.dump(
        {
            "r_160unit": float(r_unit),
            "p_160unit": float(p_unit),
            "beta_fe_clustered": float(beta),
            "p_fe_clustered": float(pval),
            "n_units": int(len(valid))
        },
        f,
        indent=2
    )

In [ ]:
# ── Cell 10: Common Layer 선택 (두 축 완료 후 1회) ──────────────
# 기준: 각 축의 layer별 최고 점수를 순위로 변환 → 순위합 최소 레이어
info_e = torch.load(vec_dir / f"economic_left_{VECTOR_TYPE}_best_layer_info.pt",
                    weights_only=False)
info_s = torch.load(vec_dir / f"social_progressive_{VECTOR_TYPE}_best_layer_info.pt",
                    weights_only=False)

layers = sorted(set(info_e["layer_scores"]) & set(info_s["layer_scores"]))
se = pd.Series({l: info_e["layer_scores"][l] for l in layers})
sc_ = pd.Series({l: info_s["layer_scores"][l] for l in layers})
rank_sum = se.rank(ascending=False) + sc_.rank(ascending=False)
COMMON_LAYER = int(rank_sum.idxmin())

print(f"axis-best: econ={info_e['best_layer']}, soci={info_s['best_layer']}")
print(f"COMMON_LAYER = {COMMON_LAYER} "
      f"(econ score={se[COMMON_LAYER]:.1f}, soci score={sc_[COMMON_LAYER]:.1f})")

torch.save({
    "model": cfg["short_name"], "vector_type": VECTOR_TYPE,
    "common_layer": COMMON_LAYER,
    "axis_best": {"economic": info_e["best_layer"],
                  "social": info_s["best_layer"]},
    "selection_rule": "min rank-sum of per-layer max steering scores",
}, vec_dir / f"common_layer_{VECTOR_TYPE}.pt")
print(f"저장: {vec_dir}/common_layer_{VECTOR_TYPE}.pt")

In [ ]:
# ── Cell 11: 두 축 기하학 (cos + layer cosine 곡선) ─────────────
# spillover steering 실험은 연구 범위에서 제외. 기하학적 관계만 보고.
vec_e_all = torch.load(vec_dir / f"economic_left_{VECTOR_TYPE}_diff.pt", weights_only=False)
vec_s_all = torch.load(vec_dir / f"social_progressive_{VECTOR_TYPE}_diff.pt", weights_only=False)
common = torch.load(vec_dir / f"common_layer_{VECTOR_TYPE}.pt", weights_only=False)
COMMON_LAYER = common["common_layer"]

v_e = vec_e_all[COMMON_LAYER].float()
v_s = vec_s_all[COMMON_LAYER].float()
cos_common = float(torch.nn.functional.cosine_similarity(
    v_e.unsqueeze(0), v_s.unsqueeze(0)))

num_layers = vec_e_all.shape[0]
layer_cos = [float(torch.nn.functional.cosine_similarity(
    vec_e_all[l].unsqueeze(0).float(), vec_s_all[l].unsqueeze(0).float()))
    for l in range(num_layers)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(num_layers), layer_cos, "o-", color="purple", markersize=4)
ax.axhline(0, color="gray", linestyle="--", alpha=0.4)
ax.axvline(COMMON_LAYER, color="red", linestyle="--", linewidth=2,
           label=f"Common layer={COMMON_LAYER} (cos={cos_common:.3f})")
ax.set_xlabel("Layer"); ax.set_ylabel("cos(v_econ, v_soci)")
ax.set_title(f"Layer-wise Cosine — {cfg['short_name']} ({VECTOR_TYPE})")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
cross_out = Path("output") / cfg["short_name"] / "cross_axis"
cross_out.mkdir(parents=True, exist_ok=True)
plt.savefig(cross_out / "layer_cosine.png", dpi=150, bbox_inches="tight")
plt.show()

with open(cross_out / "axis_geometry.json", "w") as f:
    json.dump({"model": cfg["short_name"], "vector_type": VECTOR_TYPE,
               "common_layer": COMMON_LAYER,
               "cos_at_common_layer": cos_common,
               "layer_cosines": layer_cos}, f, indent=2)
print(f"cos(v_econ, v_soci) @ layer {COMMON_LAYER} = {cos_common:.4f}")
print(f"저장: {cross_out}/axis_geometry.json")